In [1]:
pip install transformers[torch] datasets evaluate


Note: you may need to restart the kernel to use updated packages.


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# Load a pre-trained tokenizer and model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Load a dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test[:100]")
print(dataset[0])


{'text': ''}


In [2]:
import torch
import math

def calculate_perplexity(model, tokenizer, dataset):
    total_loss = 0
    total_words = 0

    for example in dataset:
        # Check if the text is empty
        if not example["text"].strip():
            continue  # Skip empty examples
        
        # Tokenize the input
        inputs = tokenizer(example["text"], return_tensors="pt", truncation=True)
        
        # Skip if the tokenized input is empty
        if inputs["input_ids"].size(1) == 0:
            continue
        
        # Perform model inference
        outputs = model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss

        # Update total loss and word count
        total_loss += loss.item() * inputs["input_ids"].size(1)  # Multiply by sequence length
        total_words += inputs["input_ids"].size(1)

    # Check to avoid division by zero
    if total_words == 0:
        raise ValueError("No valid data found for perplexity calculation.")
    
    avg_loss = total_loss / total_words
    perplexity = math.exp(avg_loss)
    return perplexity

# Example usage
# Ensure `model`, `tokenizer`, and `dataset` are defined
try:
    perplexity = calculate_perplexity(model, tokenizer, dataset)
    print(f"Perplexity: {perplexity:.2f}")
except ValueError as e:
    print(f"Error: {e}")


Perplexity: 58.05


In [3]:
from evaluate import load

# Load BLEU metric
bleu = load("bleu")

# Load or define your dataset
dataset = [
    {"text": "The quick brown fox jumps over the lazy dog."},
    {"text": "Natural language processing enables machines to understand text."},
    {"text": "The future of AI is exciting."},
    {"text": ""},
    {"text": " "}
]

# Limit the dataset to the first 10 examples for faster testing
dataset = dataset[:10]

# Function to calculate BLEU score
def calculate_bleu(model, tokenizer, dataset):
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})
        model.resize_token_embeddings(len(tokenizer))

    references = []
    predictions = []

    for example in dataset:
        if not example["text"].strip():
            continue

        inputs = tokenizer(
            example["text"], 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=512
        )

        if inputs["input_ids"].size(1) == 0:
            continue

        with torch.no_grad():
            outputs = model.generate(
                inputs["input_ids"], 
                attention_mask=inputs["attention_mask"], 
                max_length=50,
                pad_token_id=tokenizer.pad_token_id
            )

        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
        reference = example["text"]

        predictions.append(prediction)
        references.append([reference])

    if not predictions or not references:
        raise ValueError("No valid data found for BLEU score calculation.")

    bleu_score = bleu.compute(predictions=predictions, references=references)
    return bleu_score

# Example usage
try:
    bleu_score = calculate_bleu(model, tokenizer, dataset)
    print(f"BLEU Score: {bleu_score['bleu']:.4f}")
except ValueError as e:
    print(f"Error: {e}")


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


BLEU Score: 0.1679


In [6]:
from evaluate import load
import numpy as np

# Load the F1 metric
f1 = load("f1")

# Example data: Simulated true labels and predictions
# Binary classification example: 1 (positive), 0 (negative)
true_labels = [0, 0, 0, 1, 1, 1, 0, 0, 1, 1]  # Ground truth
predictions = [1, 0, 1, 1, 0, 1, 0, 0, 1, 1]   # Model predictions

# Calculate F1 Score
results = f1.compute(predictions=predictions, references=true_labels)

print(f"F1 Score: {results['f1']:.4f}")


F1 Score: 0.7273
